# Recherche du meilleur seuil de décision 

Via les notebooks d'exploration et de comparaisons des modèles, nous sommes arrivés à la conclusion qu'ici c'est un XGBoost qui permet d'obtenir les meilleurs performances pour répondre à notre problématique. Au vu des résultats obtenus sur les différentes configs testées, nous ne pensons pas pouvoir améliorer de beaucoup le modèle en tant que tel. Ce notebook à donc juste pour objectif de chercher à optimiser le seuil de décision.

In [3]:
import numpy as np
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier

from base_commune.outils import (RANDOM_STATE,charger_donnees,decouper_donnees)

In [4]:
df_brut = charger_donnees()
X_train, X_test, y_train, y_test = decouper_donnees(df_brut)
print(X_train.shape, X_test.shape)

(8000, 10) (2000, 10)


In [2]:
vars_num = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary",
            "HasCrCard", "IsActiveMember"]
vars_cate = ["Geography", "Gender"]
def construire_xgb(n_estimators=100, max_depth=6, learning_rate=0.3, scale_pos_weight=1): # On réutilise la fonction du notebook d'exploration
    preprocesseur = ColumnTransformer([
        ("cate", OneHotEncoder(handle_unknown="ignore"), vars_cate),
        ("num", "passthrough", vars_num),
    ], remainder="drop")
    return Pipeline([
        ("preprocesseur", preprocesseur),
        ("xgb", XGBClassifier(n_estimators=n_estimators, max_depth=max_depth,
                              learning_rate=learning_rate, scale_pos_weight=scale_pos_weight,
                              random_state=RANDOM_STATE, n_jobs=-1)),
    ])

modele = construire_xgb(n_estimators=300, max_depth=3, learning_rate=0.05)

In [8]:
decoupage_en_plis = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
probabilites_hors_pli = cross_val_predict(modele, X_train, y_train, cv=decoupage_en_plis, method="predict_proba")[:, 1]
precisions, rappels, seuils = precision_recall_curve(y_train, probabilites_hors_pli)
scores_f1 = 2 * precisions * rappels / (precisions + rappels + 1e-12)

indice_meilleur_f1 = np.argmax(scores_f1[:-1])
print("seuil", round(seuils[indice_meilleur_f1], 2),
      "précision", round(precisions[indice_meilleur_f1], 3),
      "rappel", round(rappels[indice_meilleur_f1], 3),
      "F1", round(scores_f1[indice_meilleur_f1], 3))


seuil 0.3 précision 0.613 rappel 0.658 F1 0.635


In [9]:
rappel_minimum = 0.70
indices_possibles = np.where(rappels[:-1] >= rappel_minimum)[0]
indice_choisi = indices_possibles[-1]   # plus grand seuil qui respecte le rappel visé
print("Rappel >=", rappel_minimum, "= seuil", round(seuils[indice_choisi], 2),
      "précision", round(precisions[indice_choisi], 3),
      "rappel", round(rappels[indice_choisi], 3),
      "F1", round(scores_f1[indice_choisi], 3))

Rappel >= 0.7 = seuil 0.25 précision 0.557 rappel 0.7 F1 0.621


# Conclusion

On voit qu'en cherchant un seuil de décision qui maximise le F1 on arrive à un rappel de 66% (2/3 des churners detectés) avec une précision correcte (61% des clients détectés comme churners le sont effectivement). 
Si on cherche à augmenter d'avantage le rappel, par exemple en fixant rappel min = 70%, la perte marginale de précision devient trop important (dans ce cas par exemple, +4 points de rappels = 6 points de précision). Si on rapporte ces "comparaisons marrginales" à notre proportion de churner/non churner on vq d'un pdv métier chercher à augmenter d'avantage le rappel n'est pas "rentable" : si on faisait tourner le modèle en prod dans un cas réel on contacterait des centaines de clients en plus pour aller capter quelques dizaines de churnes.

On va donc conserver un seuil à 0.3, qui optimise le F1 et donne ici un mix rappel/précision satisfaisant au vu de notre problématique